In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, upper, to_date, when
from pyspark.sql.types import IntegerType, DoubleType

no outlier removal because order_item_quantity has only 1-5 values

In [0]:
df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/supply_chain_data/delta/raw")

print(f"starting rows : {df.count()}")
drop_cols = [
    "Product_Description",
    "Order_Zipcode",
    "Customer_Password",
    "Customer_Email",
    "Customer_Fname",
    "Customer_Lname",
]
df = df.drop(*drop_cols)
print(f"Dropped {len(drop_cols)} columns")
print(f"Remaining: {len(df.columns)}")
df = df.fillna({
    "Customer_Zipcode": "00000"
})
print("The zipcode nulls are filled")

starting rows : 180519
Dropped 6 columns
Remaining: 47
The zipcode nulls are filled


In [0]:
string_cols = [
    "Type", "Delivery_Status", "Department_Name",
    "Category_Name", "Market", "Order_Region",
    "Order_Country", "Shipping_Mode"
]
for c in string_cols:
    if c in df.columns:
        df =  df.withColumn(c, upper(trim(col(c))))
print("standardized string columns to uppercase / trim")

standardized string columns to uppercase / trim


In [0]:
df = df.withColumn(
    "order_date_clean",
    F.coalesce(
        to_date(col("order_date_DateOrders"), "M/d/yyyy H:mm"),
        to_date(col("order_date_DateOrders"), "yyyy-MM-dd"),
        to_date(col("order_date_DateOrders"), "M/d/yyyy")
    )
)
df = df.drop("order_date_DateOrders", "shipping_date_DateOrders")
df = df.withColumn("Order_Item_Quantity", 
                   col("Order_Item_Quantity").cast(IntegerType()))
df = df.withColumn("Order_Item_Total", 
                   col("Order_Item_Total").cast(DoubleType()))
df = df.withColumn("Product_Price", 
                   col("Product_Price").cast(DoubleType()))
print("fixed date columns and cast numeric columns")

fixed date columns and cast numeric columns


In [0]:
#drop rows when null target 
before = df.count()
df = df.filter(col("Order_Item_Quantity").isNotNull())
after = df.count()
print(f"Dropped {before - after} rows with null target")

Dropped 0 rows with null target


In [0]:
print(f"\nFinal shape: {df.count()} rows × {len(df.columns)} columns")
df.select("Order_Item_Quantity").describe().show()

df.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/supply_chain_data/delta/clean")
print("Clean Delta table saved!")


Final shape: 180519 rows × 46 columns
+-------+-------------------+
|summary|Order_Item_Quantity|
+-------+-------------------+
|  count|             180519|
|   mean|  2.127637533999191|
| stddev| 1.4534514814226405|
|    min|                  1|
|    max|                  5|
+-------+-------------------+

Clean Delta table saved!
